# Project Lab 1: RAG Assistant, Reporting Agent, Safe Text-to-SQL

Notebook for **Chapter 15** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 How to Run a Project in This Area

## 15.2 Project: RAG Assistant Over Your Own Documents, With an Evaluation Harness

**Goal and why employers care.** Build an assistant that answers questions about a made-up shop (return policy, shipping FAQ, product catalog), cites the chunk it used, refuses when the documents do not contain the answer, and is scored by a harness. Almost every company wants "chat with our documents". Very few candidates show a number that proves it works. The harness is what makes this project stand out.

In [ ]:
def chunk(text, source, max_words=14):
    chunks, cur, n = [], [], 0
    for line in text.splitlines():
        if line.startswith("#"): continue          # drop the heading line
        w = len(line.split())
        if cur and n + w > max_words:
            chunks.append(" ".join(cur)); cur, n = [], 0
        cur.append(line); n += w
    if cur: chunks.append(" ".join(cur))
    return [{"id": f"{source}#{i}", "source": source, "text": t} for i, t in enumerate(chunks)]

In [ ]:
STOP = {"the","a","an","of","to","do","does","is","i","you","we","how","what","who",
        "for","in","on","and","or","it","have","are","can","long","take","much","many"}
def stem(w): return w[:-1] if len(w) > 3 and w.endswith("s") else w
def tok(s): return [stem(w) for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP]

In [ ]:
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs, self.k1, self.b = docs, k1, b
        self.toks = [tok(d["text"]) for d in docs]
        self.avg = sum(map(len, self.toks)) / len(docs)
        df = Counter(t for ts in self.toks for t in set(ts)); N = len(docs)
        self.idf = {t: math.log(1 + (N - n + .5) / (n + .5)) for t, n in df.items()}
    def search(self, q, k=3):
        if "how long" in q.lower(): q += " days business"   # tiny query expansion (step 7)
        out = []
        for d, ts in zip(self.docs, self.toks):
            tf = Counter(ts); s = 0
            for t in tok(q):
                if t in tf:
                    s += self.idf[t] * tf[t] * (self.k1 + 1) / (tf[t] + self.k1 * (1 - self.b + self.b * len(ts) / self.avg))
            out.append((s, d))
        out.sort(key=lambda x: -x[0])
        return [(round(s, 3), d) for s, d in out[:k] if s > 0]

In [ ]:
def fake_llm(question, hits, min_score=1.0):
    if not hits or hits[0][0] < min_score: return "I do not know based on the documents.", []
    qt = set(tok(question)); best = hits[0][1]
    sents = re.split(r"(?<=[.])\s+", best["text"])
    sents.sort(key=lambda s: -len(qt & set(tok(s))))
    return f"{sents[0]} [{best['id']}]", [best["id"]]

In [ ]:
good = phrase in ans and cites and cites[0].startswith(src)

## 15.3 Project: An Agent That Automates a Weekly Sales Report

**Goal and why employers care.** Build an agent that reads a shop's daily sales rows for two weeks, calls tools to compute revenue, the top product and the week-on-week change, and writes a short summary. A separate **verifier** then recomputes every number from the raw data and rejects a summary that contains an unsupported number. Companies want agents that save hours of routine work, but they will not trust one that cannot prove its numbers.

In [ ]:
def make_sales(seed=7):
    rnd = random.Random(seed); rows = [("date","product","units","unit_price")]
    prices = {"Backpack": 59, "Bottle": 18, "Lamp": 24}
    for week, start in enumerate([1, 8]):
        for d in range(start, start + 7):
            for p, price in prices.items():
                base = {"Backpack": 3, "Bottle": 8, "Lamp": 4}[p] + (2 if (week == 1 and p == "Lamp") else 0)
                rows.append((f"2026-11-{d:02d}", p, max(0, base + rnd.randint(-1, 2)), price))
    buf = io.StringIO(); csv.writer(buf).writerows(rows); return buf.getvalue()

In [ ]:
def revenue(start, end):
    return sum(int(r["units"]) * int(r["unit_price"]) for r in DATA if start <= r["date"] <= end)
def pct_change(old, new): return round((new - old) / old * 100, 1)
TOOLS = {"revenue": revenue, "top_product": top_product, "pct_change": pct_change}

In [ ]:
def run_agent(task, planner=mock_planner, max_steps=6):
    obs, log = [], []
    for step, (name, args) in enumerate(planner(task)):
        if step >= max_steps: log.append("STOPPED: step limit"); break
        if name not in TOOLS: log.append(f"REJECTED unknown tool {name}"); continue
        result = TOOLS[name](obs[0], obs[1]) if name == "pct_change" else TOOLS[name](*args)
        obs.append(result)
    ...

In [ ]:
def verify(summary):
    r1, r2 = revenue(*W1), revenue(*W2)
    truth = {str(r1), str(r2), str(pct_change(r1, r2)), top_product(*W2)["product"], str(top_product(*W2)["revenue"])}
    bad = sorted(n for n in set(re.findall(r"\d+\.?\d*", summary)) if n not in truth)
    return {"ok": not bad, "unsupported_numbers": bad}

## 15.4 Project: A Text-to-SQL Assistant With Guardrails

**Goal and why employers care.** Turn a plain-English question into SQL for a small shop database, run it safely, and show the result. The guardrails enforce read-only access, hide a private column, block multiple statements, and limit rows. The model is mocked, so you can prove that the guardrails hold even when the model misbehaves. Every analytics team wants this; the teams that ship it safely are the ones who can show tests.

In [ ]:
SCHEMA = "customers(id, name, city, email); orders(id, customer_id, order_date, total, status)"
ALLOWED_TABLES = {"customers", "orders"}
BLOCKED_COLUMNS = {"email"}                       # private data we never expose

In [ ]:
def authorizer(action, a1, a2, dbname, src):
    if action == sqlite3.SQLITE_SELECT: return sqlite3.SQLITE_OK
    if action == sqlite3.SQLITE_READ:
        if a1 not in ALLOWED_TABLES or a2 in BLOCKED_COLUMNS: return sqlite3.SQLITE_DENY
        return sqlite3.SQLITE_OK
    if action == sqlite3.SQLITE_FUNCTION: return sqlite3.SQLITE_OK
    return sqlite3.SQLITE_DENY                      # every write, DDL, pragma, attach...

In [ ]:
def guarded_run(db, sql, max_rows=100):
    s = sql.strip().rstrip(";")
    if ";" in s: return ("BLOCKED", "multiple statements")
    if not re.match(r"(?is)^\s*select\b", s): return ("BLOCKED", "only SELECT is allowed")
    db.set_authorizer(authorizer)
    try:
        return ("OK", db.execute(f"SELECT * FROM ({s}) LIMIT {max_rows}").fetchall())
    except sqlite3.DatabaseError as e:
        blocked = "not authorized" in str(e) or "prohibited" in str(e)
        return ("BLOCKED" if blocked else "ERROR", str(e))
    finally:
        db.set_authorizer(None)

## 15.5 Checkpoints, Grading Rubric and Five Debugging Problems

Projects go wrong in quiet ways: the code runs, the number looks fine, and the system is broken. This section gives you checkpoints (what you should see after each step), a rubric (how a reviewer would score your lab), and five bugs to find and fix. Every snippet runs here, and the output shown is real.

In [ ]:
import re
filler = lambda n, tag: " ".join(f"{tag}{i}" for i in range(n))
doc = filler(37, "a") + " the refund window is thirty days from delivery " + filler(60, "b")
words = doc.split()
def chunk(size, overlap):
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, len(words), step)]
query = {"refund", "window", "days"}
for size, overlap in [(40, 0), (20, 0), (20, 10), (60, 0)]:
    cs = chunk(size, overlap)
    best = max(cs, key=lambda c: len(query & set(c.split())))
    print(f"size {size:2d} overlap {overlap:2d}: chunks {len(cs):2d}, top chunk has full answer: {'thirty days' in best}")

size 40 overlap  0: chunks  3, top chunk has full answer: False
size 20 overlap  0: chunks  6, top chunk has full answer: False
size 20 overlap 10: chunks 11, top chunk has full answer: True
size 60 overlap  0: chunks  2, top chunk has full answer: True


In [ ]:
results = [("c1", 0.91, "Refunds take 5 days."), ("c1b", 0.91, "Refunds take 5 days."),
           ("c1c", 0.90, "Refunds take 5 days."), ("c2", 0.74, "Refund window is 30 days."),
           ("c3", 0.70, "Delivery is free over 50.")]
print("top 3 as returned:", [r[0] for r in results[:3]])
seen, unique = set(), []
for cid, score, text in results:
    key = text.lower().strip()
    if key not in seen:
        seen.add(key); unique.append(cid)
print("after dedupe:     ", unique[:3])

top 3 as returned: ['c1', 'c1b', 'c1c']
after dedupe:      ['c1', 'c2', 'c3']


In [ ]:
def planner(history):                       # a stuck mock model: always asks the same thing
    return ("search", "refund policy")
history, calls = [], {}
for step in range(1, 7):
    call = planner(history)
    calls[call] = calls.get(call, 0) + 1
    if calls[call] > 2:
        print(f"step {step}: stop, call {call} repeated {calls[call]} times")
        break
    history.append(call)
    print(f"step {step}: {call}")

step 1: ('search', 'refund policy')
step 2: ('search', 'refund policy')
step 3: stop, call ('search', 'refund policy') repeated 3 times


In [ ]:
import re, sqlite3
def weak_guard(sql):
    return sql.strip().lower().startswith("select")
def strong_guard(sql):
    s = sql.strip().rstrip(";")
    if ";" in s or "--" in s or "/*" in s: return False
    return re.match(r"(?is)^select\b", s) is not None and not re.search(r"(?i)\b(insert|update|delete|drop|alter|attach|pragma)\b", s)
tests = ["SELECT * FROM orders", "select 1; DROP TABLE orders", "SELECT * FROM orders -- ; harmless", "DROP TABLE orders", "SELECT 1;"]
for t in tests:
    print(f"weak={weak_guard(t)!s:5} strong={strong_guard(t)!s:5} {t}")
db = sqlite3.connect(":memory:"); db.execute("create table orders(id int)")
try:
    db.executescript("select 1; DROP TABLE orders")
    print("tables left:", db.execute("select count(*) from sqlite_master where name='orders'").fetchone()[0])
except Exception as e: print("error", e)

weak=True  strong=True  SELECT * FROM orders
weak=True  strong=False select 1; DROP TABLE orders
weak=True  strong=False SELECT * FROM orders -- ; harmless
weak=False strong=False DROP TABLE orders
weak=True  strong=True  SELECT 1;
tables left: 0


In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
create table orders(id int, customer text, total real);
create table items(order_id int, sku text);
insert into orders values (1, 'asha', 100), (2, 'ravi', 50);
insert into items values (1, 'A'), (1, 'B'), (2, 'A');
""")
print("wrong:", db.execute("select sum(o.total) from orders o join items i on i.order_id = o.id").fetchone()[0])
print("right:", db.execute("select sum(total) from orders").fetchone()[0])
print("right with items filter:", db.execute("select sum(total) from orders where id in (select order_id from items where sku = 'A')").fetchone()[0])

wrong: 250.0
right: 150.0
right with items filter: 150.0


## Chapter recap

## Practice set

**Task 1 (easy).** The RAG eval has 7 answerable questions and 6 have the right source in the top 3. What is recall@3? **Answer:** 6 / 7 = 0.857, about 86 percent.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch15.html